## Cálculo dos IS's

Esse notebook cuida do cálculo dos índices: **IS Ambiental** (vida útil),
**IS Energético** (etiqueta + queda brusca) e **IS Financeiro** (aproximação,
energia restante vs. investimento).

Depende do notebook de conexão (`apollo_connection_neon_core`), chamado via
`%run` na primeira célula.

**Decisões de cálculo definidas pelo grupo:**
1. IS Ambiental = só a vida útil restante do painel, em %
2. IS Energético: peso 50/50 entre "abaixo da etiqueta" e "queda brusca"
3. "Queda brusca" é um score contínuo (não um corte fixo tipo "caiu mais de X%")
4. IS Financeiro é uma APROXIMAÇÃO (kWh restante ÷ investimento do painel/lote),
   não o cálculo real com tarifa e conta de luz — isso precisa ficar claro
   pro 1º ano na apresentação, não é a definição original deles
5. Agregação painel → lote → filial: média simples em todos os níveis
6. Painel sem medição anterior recebe componente de estabilidade máximo (1.0)

In [0]:
%run "./apollo_connection_neon_core"

### Mapeamento Lote -> Filial

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_batch_company_unit AS
SELECT DISTINCT batch_id, company_unit_id
FROM stg_panels_batch;

### IS Ambiental - Painel, Lote e Filial

In [0]:
%sql
-- PAINEL
CREATE OR REPLACE TEMPORARY VIEW vw_panel_vida_util AS
SELECT
  panel_id,
  batch_id,
  estimated_life_cycle,
  GREATEST(estimated_life_cycle - (DATEDIFF(CURRENT_DATE(), activated_at) / 365.0), 0) AS years_remaining,
  GREATEST(estimated_life_cycle - (DATEDIFF(CURRENT_DATE(), activated_at) / 365.0), 0)
    / estimated_life_cycle AS vu
FROM stg_panels_batch;

In [0]:
%sql
-- LOTE
CREATE OR REPLACE TEMPORARY VIEW vw_batch_is_ambiental AS
SELECT batch_id, ROUND(AVG(vu) * 100, 2) AS is_ambiental
FROM vw_panel_vida_util
GROUP BY batch_id;

In [0]:
%sql
-- FILIAL
CREATE OR REPLACE TEMPORARY VIEW vw_company_unit_is_ambiental AS
SELECT bcu.company_unit_id, ROUND(AVG(b.is_ambiental), 2) AS is_ambiental
FROM vw_batch_is_ambiental b
JOIN vw_batch_company_unit bcu ON bcu.batch_id = b.batch_id
GROUP BY bcu.company_unit_id;

### IS Financeiro - Painel, Lote e Filial

In [0]:
%sql
-- PAINEL
CREATE OR REPLACE TEMPORARY VIEW vw_avg_generation AS
SELECT panel_id, AVG(generated_energy_kwh) AS avg_daily_generation_kwh
FROM stg_measurements
GROUP BY panel_id;

CREATE OR REPLACE TEMPORARY VIEW vw_panel_remaining_energy AS
SELECT
  vu.panel_id,
  vu.batch_id,
  ag.avg_daily_generation_kwh * 365 * vu.years_remaining AS remaining_energy_kwh
FROM vw_panel_vida_util vu
LEFT JOIN vw_avg_generation ag ON ag.panel_id = vu.panel_id;

CREATE OR REPLACE TEMPORARY VIEW vw_panel_rf_raw AS
SELECT
  re.panel_id,
  re.batch_id,
  re.remaining_energy_kwh / p.unit_cost AS rf_raw
FROM vw_panel_remaining_energy re
JOIN stg_panels_batch p ON p.panel_id = re.panel_id;

In [0]:
%sql
-- LOTE
CREATE OR REPLACE TEMPORARY VIEW vw_batch_rf_raw AS
SELECT batch_id, AVG(rf_raw) AS rf_raw_mean
FROM vw_panel_rf_raw
GROUP BY batch_id;

CREATE OR REPLACE TEMPORARY VIEW vw_batch_is_financeiro AS
SELECT
  batch_id,
  ROUND(
    (rf_raw_mean - MIN(rf_raw_mean) OVER ()) / (MAX(rf_raw_mean) OVER () - MIN(rf_raw_mean) OVER ()) * 100,
    2
  ) AS is_financeiro
FROM vw_batch_rf_raw;

In [0]:
%sql
-- FILIAL
CREATE OR REPLACE TEMPORARY VIEW vw_company_unit_is_financeiro AS
SELECT bcu.company_unit_id, ROUND(AVG(b.is_financeiro), 2) AS is_financeiro
FROM vw_batch_is_financeiro b
JOIN vw_batch_company_unit bcu ON bcu.batch_id = b.batch_id
GROUP BY bcu.company_unit_id;

### IS Energético - Painel, Lote E Filal

In [0]:
%sql
-- PAINEL
CREATE OR REPLACE TEMPORARY VIEW vw_panel_efficiency_history AS
SELECT
  panel_id,
  current_efficiency,
  measured_at,
  ROW_NUMBER() OVER (PARTITION BY panel_id ORDER BY measured_at DESC) AS rn
FROM stg_measurements;

CREATE OR REPLACE TEMPORARY VIEW vw_latest_efficiency AS
SELECT panel_id, current_efficiency AS latest_efficiency
FROM vw_panel_efficiency_history
WHERE rn = 1;

CREATE OR REPLACE TEMPORARY VIEW vw_previous_avg_efficiency AS
SELECT panel_id, AVG(current_efficiency) AS avg_previous_efficiency
FROM vw_panel_efficiency_history
WHERE rn > 1
GROUP BY panel_id;

CREATE OR REPLACE TEMPORARY VIEW vw_panel_is_energetico AS
SELECT
  p.panel_id,
  p.batch_id,
  LEAST(le.latest_efficiency / p.rated_efficiency, 1.0) AS e_ratio,
  CASE
    WHEN pa.avg_previous_efficiency IS NULL THEN 1.0
    ELSE GREATEST(
      1.0 - GREATEST((pa.avg_previous_efficiency - le.latest_efficiency) / pa.avg_previous_efficiency, 0),
      0
    )
  END AS stability_score
FROM stg_panels_batch p
LEFT JOIN vw_latest_efficiency le ON le.panel_id = p.panel_id
LEFT JOIN vw_previous_avg_efficiency pa ON pa.panel_id = p.panel_id;

In [0]:
%sql
-- LOTE
CREATE OR REPLACE TEMPORARY VIEW vw_batch_is_energetico AS
SELECT
  batch_id,
  ROUND((0.5 * AVG(e_ratio) + 0.5 * AVG(stability_score)) * 100, 2) AS is_energetico
FROM vw_panel_is_energetico
GROUP BY batch_id;

In [0]:
%sql
-- FILIAL
CREATE OR REPLACE TEMPORARY VIEW vw_company_unit_is_energetico AS
SELECT bcu.company_unit_id, ROUND(AVG(b.is_energetico), 2) AS is_energetico
FROM vw_batch_is_energetico b
JOIN vw_batch_company_unit bcu ON bcu.batch_id = b.batch_id
GROUP BY bcu.company_unit_id;

### IS Geral - Lote

In [0]:
%sql
CREATE OR REPLACE TABLE apollo_bi.fact_batch_is AS
SELECT
  f.batch_id,
  a.is_ambiental,
  e.is_energetico,
  f.is_financeiro,
  CURRENT_TIMESTAMP() AS calculated_at
FROM vw_batch_is_financeiro f
JOIN vw_batch_is_ambiental a ON a.batch_id = f.batch_id
JOIN vw_batch_is_energetico e ON e.batch_id = f.batch_id;

In [0]:
%sql
CREATE OR REPLACE TABLE apollo_bi.fact_company_unit_is AS
SELECT
  a.company_unit_id,
  cu.unit_name AS filial_nome,
  a.is_ambiental,
  e.is_energetico,
  f.is_financeiro,
  ROUND(0.40 * f.is_financeiro + 0.30 * a.is_ambiental + 0.30 * e.is_energetico, 2) AS is_geral,
  CURRENT_TIMESTAMP() AS calculated_at
FROM vw_company_unit_is_ambiental a
JOIN vw_company_unit_is_energetico e ON e.company_unit_id = a.company_unit_id
JOIN vw_company_unit_is_financeiro f ON f.company_unit_id = a.company_unit_id
JOIN stg_company_unit cu ON cu.company_unit_key = a.company_unit_id;

In [0]:
%sql
SELECT * FROM apollo_bi.fact_company_unit_is ORDER BY filial_nome;